# Patch SAE segmentation: inference sweep and paper tables

Enable **Internet** and select **T4 GPU**. No Hugging Face account, token or upload.
Public pretrained models/datasets are downloaded; results stay in Kaggle.
Push the updated code to GitHub `refactor` before running.

SigLIP 2, DINOv2 and CLIP on VOC and ADE20K: **18 SAE fits at train K=12**,
**60 evaluations** (54 sparse at inference K=3,12,48 plus 6 dense).
Existing fits and K=12 evaluations are reused in the same retained workspace:
only **36 additional evaluations**, no additional fitting. A new empty workspace
needs the original fitting/extraction again. The previous K=12-only config is
archived as `configs/patch_scene_kaggle_k12.yaml`.

The default selects all six jobs sequentially. For a shorter session, select one
encoder and dataset below (3 fits, 10 evaluations). This does not assume that all
six jobs fit one session. Download the refreshed results ZIP before the session ends.


In [ ]:
ENCODER = "all"  # "all", "dinov2", "siglip2", "clip"
DATASET = "all"  # "all", "voc", "ade20k"
GIT_REPO = "https://github.com/thienquang204/MPSAE.git"
GIT_BRANCH = "refactor"


In [ ]:
import os, sys, subprocess, json
from pathlib import Path

# Public downloads only. No login or secret lookup.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = "/tmp/mpsae-hf"
os.environ["HF_DATASETS_CACHE"] = "/tmp/mpsae-datasets"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
REPO = Path("/kaggle/working/MPSAE")
WORK = Path("/kaggle/working/patch-scene-v2")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, "--depth", "1", GIT_REPO, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["nvidia-smi"], check=True)
assert (REPO / "mpsae/patch_scene/__main__.py").exists(), "Push the new code to GitHub first."
assert "def export_results" in (REPO / "mpsae/patch_scene/report.py").read_text(), "Push the local-results update first."
assert "def inference_ks" in (REPO / "mpsae/patch_scene/config.py").read_text(), "Push the inference-sweep update first."


## Isolated environment
Uses Python 3.11 and CUDA 12.6 PyTorch, independently of Kaggle's installed
Python/packages. P100 is unsupported by the project guard. T4 x2 uses GPU 0.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
PYTHON = "/tmp/mpsae-env/bin/python"
if not Path(PYTHON).exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", "/tmp/mpsae-env"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "torch==2.9.1", "torchvision==0.24.1",
                "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--no-cache-dir",
                "-r", "requirements-patch-scene.txt"], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "doctor", "--device", "cuda:0"], check=True)


In [ ]:
CONFIG = "configs/patch_scene_kaggle.yaml"
PLAN = json.loads(subprocess.check_output([
    PYTHON, "-m", "mpsae.patch_scene", "plan", "--config", CONFIG], text=True))
print(json.dumps(PLAN, indent=2))
assert PLAN["total_evaluations"] == 60 and PLAN["inference_k"] == [3, 12, 48]
assert PLAN["sae_fits"] == 18 and PLAN["train_k"] == 12
jobs = (len(PLAN["encoders"]) if ENCODER == "all" else 1) * (len(PLAN["datasets"]) if DATASET == "all" else 1)
print(f"Selected: {DATASET}/{ENCODER}; {jobs} jobs, {jobs*10} evaluations (completed work is reused).")


## Run or resume locally
All requested encoders/datasets run sequentially, with one encoder resident at a
time. Each SAE is fitted once and reused at all three inference K values.
The dense reference is evaluated once per dataset/encoder. Training remains K=12.
Rerun with the same retained workspace to resume; no login or upload is needed.

The automatic ZIP is at `/kaggle/working/patch-scene-v2/patch-scene-results.zip`.
Completed results are kept even if a later evaluation fails. When this cell exits,
the ZIP is also copied to `/kaggle/working/patch-scene-results.zip`.


In [ ]:
import shutil

command = [PYTHON, "-u", "-m", "mpsae.patch_scene", "run",
           "--config", CONFIG, "--encoder", ENCODER, "--dataset", DATASET,
           "--workspace", str(WORK), "--device", "cuda:0"]
try:
    subprocess.run(command, check=True)
finally:
    # Preserve already completed results even if a later method raises an error.
    saved = WORK / "patch-scene-results.zip"
    if saved.exists():
        shutil.copy2(saved, "/kaggle/working/patch-scene-results.zip")
        print("Download: /kaggle/working/patch-scene-results.zip")


In [ ]:
from IPython.display import FileLink, display

# This cell may also be run after interrupting the training cell.
# Export only combines completed local results; it does not train or evaluate.
subprocess.run([PYTHON, "-m", "mpsae.patch_scene", "export",
                "--config", CONFIG, "--workspace", str(WORK)], check=True)
archive = Path("/kaggle/working/patch-scene-results.zip")
import shutil
shutil.copy2(WORK / "patch-scene-results.zip", archive)
table = WORK / "campaigns" / PLAN["campaign"] / "table"
receipt = json.loads((table / "results.json").read_text())
print(f"Completed: {receipt['completed']}/{receipt['expected']}")
import pandas as pd
display(pd.read_csv(table / "comparisons.csv").round(2))
print("LaTeX: results.tex, table_voc.tex, table_ade20k.tex, pixel_accuracy.tex")
# Use a relative link with the notebook working directory at /kaggle/working.
previous = Path.cwd()
try:
    os.chdir(archive.parent)
    display(FileLink(archive.name))
finally:
    os.chdir(previous)
print("Or use Kaggle's file browser: /kaggle/working/patch-scene-results.zip -> Download")


## Download and preserve
Download **patch-scene-results.zip** from the final cell or Kaggle's file browser.
It includes CSV/JSON, settings, mIoU and pixel-accuracy LaTeX tables, and a standalone
`tables_preview.tex` document. Use `\usepackage{booktabs}` then `\input{results.tex}`
in your paper; `table_voc.tex` and `table_ade20k.tex` provide separate tables.

Tables show inference K and MPSAE gains in **percentage points** over the stronger
CSR/CSRv2 baseline. Bold marks the best sparse score per row only when all methods
are available; ties are retained. Missing entries are `--`. Measured nonzero counts
are in the CSV. Single-seed results do not establish statistical significance.
CSRv2 uses training-mask labels. These are dataset-specific SAEs, not a cross-dataset
transfer experiment. No published reference curves or figures are generated.

There is no remote backup. Download the ZIP before ending the session, or save a
notebook version **with output files**. The results ZIP excludes checkpoints and
features. Restore the **full workspace** to reuse them in a later session.
See `docs/patch_scene_kaggle.md` for details.
